# DermaBench: Evaluation-First Skin Lesion Classification
**Subgroup, robustness and calibration analysis on DermaMNIST (MedMNIST+)**

The goal of this notebook is **not** to chase a leaderboard accuracy. It is to practise (and show) how an AI/ML algorithm component is *scientifically evaluated* the way a regulated medical-AI team would:

1. Define the **intended use and acceptance criteria *before* seeing test results**
2. Audit the data (class imbalance, image-quality proxy subgroups, provenance fingerprints)
3. Train two models (CNN vs ViT) reproducibly, tracked with MLflow
4. Evaluate: per-class metrics with bootstrap CIs, **calibration** (+ temperature scaling), **proxy-subgroup** performance, **robustness** to corruptions, **failure modes** (EigenCAM / Grad-CAM)
5. Report against the acceptance criteria (pass **or** fail, honestly) and auto-generate a **model card**

> **Intended use (for framing only):** research/education benchmark for a decision-support *triage* component. **Not a medical device, not validated for clinical use.**

**How to run (Kaggle):** right sidebar → Settings → **Accelerator: GPU T4 x2 or P100**, and **Internet: On** (needed for `pip`, the MedMNIST download and pretrained timm weights; Kaggle asks for phone verification the first time). Then *Run All*. Expect ~15-25 min at 224 px. Outputs are saved in `/kaggle/working` and appear under the notebook's **Output** tab.

## 1. Setup

In [ ]:
# @title Install dependencies
!pip -q install medmnist timm mlflow grad-cam scikit-learn scipy pandas matplotlib tabulate

## 2. Configuration
All knobs live here. Change `img_size` to 64 or 128 for faster runs (224 gives the most meaningful CAM visualisations).

In [ ]:
import os, json, random, time, io, math, warnings, hashlib, platform, sys, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy import ndimage, optimize
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
warnings.filterwarnings("ignore")

SMOKE_TEST = False   # developer flag: tiny synthetic run to test the pipeline logic. Keep False in Colab.

CFG = dict(
    seed=42,
    img_size=224,                                   # 28 | 64 | 128 | 224 (MedMNIST+)
    models=["resnet50", "vit_small_patch16_224"],   # timm model names
    epochs=10, batch_size=64, lr=1e-4, weight_decay=1e-4,
    num_workers=2,
    n_boot=500,            # bootstrap resamples for confidence intervals
    ece_bins=15,
    min_subgroup_n=50,     # subgroups smaller than this are reported but excluded from gap criteria
    min_class_n_subgroup=15,
    reuse_checkpoints=True # if a checkpoint exists (e.g. after a Colab disconnect) skip retraining
)
if SMOKE_TEST:
    CFG.update(img_size=64, models=["resnet18", "vit_tiny_patch16_224"], epochs=2,
               batch_size=32, num_workers=0, n_boot=50, reuse_checkpoints=False)

OUT = "/kaggle/working/dermabench_outputs" if os.path.isdir("/kaggle/working") else "./dermabench_outputs"
for sub in ["figures", "tables", "checkpoints", "mlruns"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])

ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
           torch=str(torch.__version__), device=DEVICE,
           gpu=(torch.cuda.get_device_name(0) if DEVICE == "cuda" else None))
for pkg in ["timm", "sklearn", "scipy", "mlflow", "medmnist"]:
    try:
        ENV[pkg] = __import__(pkg).__version__
    except Exception:
        pass
json.dump(dict(config=CFG, environment=ENV), open(f"{OUT}/tables/run_config_environment.json", "w"), indent=2, default=str)
print("Device:", DEVICE, "|", ENV.get("gpu"))
print(json.dumps(CFG, indent=2))

## 3. Intended use & acceptance criteria (set these **before** looking at test results)
A scientific evaluation fixes its pass/fail criteria up front. The values below are **illustrative defaults**: edit them to what *you* can justify, then **don't change them after seeing test results**. The cell timestamps them into `criteria_frozen.json`. A failed criterion is a legitimate, reportable finding.

| Criterion | Meaning |
|---|---|
| `macro_auroc_min` | One-vs-rest macro AUROC across the 7 classes |
| `mel_sensitivity_min` | Melanoma sensitivity (the clinically costly miss) |
| `ece_after_ts_max` | Calibration error after temperature scaling |
| `max_subgroup_balacc_gap` | Largest balanced-accuracy gap between overall and any image-quality proxy subgroup |
| `max_moderate_mel_sens_drop` | Largest absolute drop in melanoma sensitivity under *moderate* corruptions |

In [ ]:
CRITERIA = dict(
    macro_auroc_min=0.90,
    mel_sensitivity_min=0.60,
    ece_after_ts_max=0.05,
    max_subgroup_balacc_gap=0.10,
    max_moderate_mel_sens_drop=0.15,
)
frozen = dict(criteria=CRITERIA, frozen_at_utc=datetime.datetime.utcnow().isoformat(timespec="seconds"),
              intended_use="Research benchmark for a skin-lesion triage decision-support component. Not a medical device.")
json.dump(frozen, open(f"{OUT}/tables/criteria_frozen.json", "w"), indent=2)
print(json.dumps(frozen, indent=2))

## 4. Data: load, audit, fingerprint
DermaMNIST is the MedMNIST version of HAM10000 (10,015 dermoscopy images, 7 classes) with fixed train/val/test splits.
**Limitation to remember:** MedMNIST does not expose patient/lesion IDs or demographics, so we cannot verify that splits are lesion-disjoint and cannot do demographic subgroup analysis. We use *image-quality proxy subgroups* instead and say so in the report.

In [ ]:
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_LONG = ["actinic keratoses / intraepithelial carcinoma", "basal cell carcinoma", "benign keratosis-like lesions",
              "dermatofibroma", "melanoma", "melanocytic nevi", "vascular lesions"]
K, MEL = 7, 4

def load_data():
    data = {}
    if SMOKE_TEST:  # synthetic, weakly learnable data so the whole pipeline can be exercised offline
        rng = np.random.default_rng(0)
        for split, n in [("train", 350), ("val", 140), ("test", 210)]:
            y = np.arange(n) % K
            X = rng.integers(0, 256, (n, CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)
            X = np.clip(X * 0.5 + y[:, None, None, None] * 30, 0, 255).astype(np.uint8)
            data[split] = (X, y.astype(int))
        return data
    from medmnist import DermaMNIST, INFO
    labels = INFO["dermamnist"]["label"]
    assert "melanoma" in labels["4"].lower(), f"Unexpected label map: {labels}"
    for split in ["train", "val", "test"]:
        ds = DermaMNIST(split=split, download=True, size=CFG["img_size"], as_rgb=True)
        data[split] = (np.asarray(ds.imgs, dtype=np.uint8), np.asarray(ds.labels).squeeze().astype(int))
    return data

DATA = load_data()
Xtr, ytr = DATA["train"]; Xva, yva = DATA["val"]; Xte, yte = DATA["test"]

# --- provenance fingerprint: lets anyone verify they evaluated on the same data
fingerprint = {s: dict(n=int(len(DATA[s][1])), shape=list(DATA[s][0].shape),
                       sha256_images=hashlib.sha256(DATA[s][0].tobytes()).hexdigest()[:16],
                       sha256_labels=hashlib.sha256(DATA[s][1].astype(np.int64).tobytes()).hexdigest()[:16])
               for s in DATA}
json.dump(fingerprint, open(f"{OUT}/tables/data_fingerprint.json", "w"), indent=2)
print(json.dumps(fingerprint, indent=2))

# --- class distribution audit
dist = pd.DataFrame({s: np.bincount(DATA[s][1], minlength=K) for s in DATA}, index=CLASS_NAMES)
dist["total"] = dist.sum(1); dist["pct_train"] = (100 * dist["train"] / dist["train"].sum()).round(1)
dist.to_csv(f"{OUT}/tables/class_distribution.csv"); display(dist)
print(f"Imbalance ratio (largest/smallest class in train): {dist['train'].max() / dist['train'].min():.1f}x")

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/figures/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
dist[["train", "val", "test"]].plot.bar(ax=axes[0]); axes[0].set_title("Class counts per split"); axes[0].set_yscale("log")
cols = []
for c in range(K):
    cols.append(Xtr[np.where(ytr == c)[0][0]])
axes[1].imshow(np.concatenate(cols, axis=1)); axes[1].axis("off")
axes[1].set_title("One training example per class: " + ", ".join(CLASS_NAMES), fontsize=9)
savefig("01_data_audit")

### 4b. Image-quality proxy subgroups
No demographics are available, so we derive **image-derived proxy features** (brightness, contrast, sharpness, colourfulness) and split them into low/mid/high bands using **training-set tertiles**. This checks whether the model quietly degrades on darker, flatter or blurrier images. These are **not** patient subgroups and must never be described as fairness across demographics.

In [ ]:
FEATURES = ["brightness", "contrast", "sharpness", "colorfulness"]

def image_features(X):
    w = np.array([0.299, 0.587, 0.114], dtype=np.float32)
    rows = []
    for im in X:
        f = im.astype(np.float32); g = f @ w
        rg = f[..., 0] - f[..., 1]; yb = 0.5 * (f[..., 0] + f[..., 1]) - f[..., 2]
        colorful = np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2)
        rows.append((g.mean(), g.std(), ndimage.laplace(g).var(), colorful))
    return pd.DataFrame(rows, columns=FEATURES)

feat_tr = image_features(Xtr)
THRESH = {f: np.quantile(feat_tr[f], [1 / 3, 2 / 3]).tolist() for f in FEATURES}
json.dump(THRESH, open(f"{OUT}/tables/proxy_subgroup_thresholds_from_train.json", "w"), indent=2)

def assign_bands(feat_df):
    out = pd.DataFrame(index=feat_df.index)
    for f in FEATURES:
        out[f + "_band"] = np.array(["low", "mid", "high"])[np.digitize(feat_df[f], THRESH[f])]
    return out

feat_te = image_features(Xte)
BANDS_TE = assign_bands(feat_te)
display(BANDS_TE.apply(lambda s: s.value_counts()).T)

## 5. Helper library: metrics, calibration, perturbations
Everything the evaluation uses is defined here and unit-tested in the next cell.

In [ ]:
def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def ece_score(probs, y, n_bins=None):
    n_bins = n_bins or CFG["ece_bins"]
    conf = probs.max(1); acc = (probs.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def nll_score(p, y): return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())
def brier_score(p, y): return float(((p - np.eye(K)[y]) ** 2).sum(1).mean())

def sens_spec_prec(pred, y, c):
    tp = ((pred == c) & (y == c)).sum(); fn = ((pred != c) & (y == c)).sum()
    tn = ((pred != c) & (y != c)).sum(); fp = ((pred == c) & (y != c)).sum()
    return tp / max(tp + fn, 1), tn / max(tn + fp, 1), tp / max(tp + fp, 1)

METRIC_FNS = {
    "acc":         lambda p, y: float((p.argmax(1) == y).mean()),
    "bal_acc":     lambda p, y: float(balanced_accuracy_score(y, p.argmax(1))),
    "macro_auroc": lambda p, y: float(roc_auc_score(y, p, multi_class="ovr", average="macro", labels=list(range(K)))),
    "mel_auroc":   lambda p, y: float(roc_auc_score((y == MEL).astype(int), p[:, MEL])),
    "mel_sens":    lambda p, y: float(sens_spec_prec(p.argmax(1), y, MEL)[0]),
    "mel_spec":    lambda p, y: float(sens_spec_prec(p.argmax(1), y, MEL)[1]),
    "ece":         lambda p, y: ece_score(p, y),
    "nll":         lambda p, y: nll_score(p, y),
    "brier":       lambda p, y: brier_score(p, y),
}

def core_metrics(probs, y): return {k: fn(probs, y) for k, fn in METRIC_FNS.items()}

def bootstrap_ci(fn, probs, y, n_boot, seed=0):
    rng = np.random.default_rng(seed); N = len(y); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, N, N)
        try: vals.append(fn(probs[idx], y[idx]))
        except ValueError: pass   # resample missing a class -> skip
    return tuple(np.percentile(vals, [2.5, 97.5])) if len(vals) > 10 else (np.nan, np.nan)

def summary_table(probs, y, n_boot):
    rows = []
    for k, fn in METRIC_FNS.items():
        lo, hi = bootstrap_ci(fn, probs, y, n_boot)
        rows.append(dict(metric=k, value=fn(probs, y), ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)

def per_class_table(probs, y):
    pred = probs.argmax(1); rows = []
    for c in range(K):
        se, sp, pr = sens_spec_prec(pred, y, c)
        f1 = 2 * pr * se / max(pr + se, 1e-12)
        au = roc_auc_score((y == c).astype(int), probs[:, c]) if (y == c).any() else np.nan
        rows.append(dict(cls=CLASS_NAMES[c], n=int((y == c).sum()), sensitivity=se, specificity=sp,
                         precision=pr, f1=f1, auroc_ovr=au))
    return pd.DataFrame(rows)

def fit_temperature(logits, y):
    # single-parameter temperature scaling fit by minimising validation NLL (scipy, deterministic)
    logits = np.asarray(logits, dtype=np.float64)
    f = lambda logT: nll_score(softmax_np(logits / np.exp(logT)), y)
    res = optimize.minimize_scalar(f, bounds=(-3, 3), method="bounded")
    return float(np.exp(res.x))

# ---------------- perturbations (operate on uint8 N,H,W,3 arrays) ----------------
def _chunks(X, fn, chunk=256):
    return np.concatenate([fn(X[i:i + chunk]) for i in range(0, len(X), chunk)])

def perturb(X, kind, level, seed=0):
    rng = np.random.default_rng(seed)
    if kind == "gaussian_noise":
        f = lambda c: np.clip(c.astype(np.float32) + rng.normal(0, level * 255, c.shape).astype(np.float32), 0, 255).astype(np.uint8)
    elif kind == "gaussian_blur":
        s = level * X.shape[1] / 224.0
        f = lambda c: np.clip(ndimage.gaussian_filter(c.astype(np.float32), sigma=(0, s, s, 0)), 0, 255).astype(np.uint8)
    elif kind == "brightness_down":
        f = lambda c: np.clip(c.astype(np.float32) * level, 0, 255).astype(np.uint8)
    elif kind == "contrast_down":
        def f(c):
            c = c.astype(np.float32); m = c.mean(axis=(1, 2, 3), keepdims=True)
            return np.clip((c - m) * level + m, 0, 255).astype(np.uint8)
    elif kind == "jpeg":
        from PIL import Image
        def f(c):
            out = []
            for im in c:
                buf = io.BytesIO(); Image.fromarray(im).save(buf, format="JPEG", quality=int(level)); buf.seek(0)
                out.append(np.array(Image.open(buf).convert("RGB")))
            return np.stack(out)
    elif kind == "drop_channel":
        idx = {"R": 0, "G": 1, "B": 2}[level]
        def f(c):
            c = c.copy(); c[..., idx] = 0; return c
    else:
        raise ValueError(kind)
    return _chunks(X, f)

LEVEL_NAMES = ["mild", "moderate", "severe"]
PERTURBATIONS = {                    # kind -> three severity parameters
    "gaussian_noise":  [0.02, 0.05, 0.10],   # sigma as a fraction of 255
    "gaussian_blur":   [1, 2, 4],            # sigma in px at 224 (scaled with resolution)
    "brightness_down": [0.8, 0.6, 0.4],      # multiplicative factor
    "contrast_down":   [0.7, 0.5, 0.3],      # multiplicative factor around the image mean
    "jpeg":            [50, 20, 5],          # JPEG quality
}
CHANNELS = ["R", "G", "B"]   # 'missing channel' = a crude stand-in for a missing input modality

### 5b. Sanity tests of the helper library
Small unit tests so the evaluation code itself is verified before it is trusted.

In [ ]:
y_ = np.arange(70) % K
assert ece_score(np.eye(K)[y_], y_) < 1e-9, "ECE of perfect predictions should be 0"
assert ece_score(np.eye(K)[(y_ + 1) % K], y_) > 0.99, "ECE of confidently-wrong predictions should be ~1"

rng_ = np.random.default_rng(0)
yy = rng_.integers(0, K, 2000); zz = rng_.normal(size=(2000, K)); zz[np.arange(2000), yy] += 1.5; zz *= 5   # overconfident logits
T_ = fit_temperature(zz, yy)
assert T_ > 1.0, "overconfident logits should need T>1"
assert nll_score(softmax_np(zz / T_), yy) < nll_score(softmax_np(zz), yy), "temperature scaling must reduce NLL"

pp = softmax_np(zz / T_); est = METRIC_FNS["bal_acc"](pp, yy); lo, hi = bootstrap_ci(METRIC_FNS["bal_acc"], pp, yy, 100)
assert lo <= est <= hi, "bootstrap CI should contain the point estimate"

Xs = rng_.integers(0, 256, (20, CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)
for kind, levels in PERTURBATIONS.items():
    for lv in levels:
        Xp = perturb(Xs, kind, lv)
        assert Xp.shape == Xs.shape and Xp.dtype == np.uint8, (kind, lv)
        assert (Xp != Xs).any(), f"{kind}@{lv} did not change the image"
for ch in CHANNELS:
    assert (perturb(Xs, "drop_channel", ch)[..., "RGB".index(ch)] == 0).all()
assert (perturb(Xs, "brightness_down", 0.5).mean() < Xs.mean())
assert image_features(Xs).shape == (20, 4)
print("All helper-library sanity tests passed ✅")

## 6. Models & training
Two fine-tuned ImageNet-pretrained models via `timm`: a **CNN (ResNet-50)** and a **Vision Transformer (ViT-S/16)**. Class-weighted loss (square-root inverse frequency) to cope with imbalance, light geometric augmentation, AdamW + cosine schedule, mixed precision. **The checkpoint is selected on validation macro-AUROC, never on the test set.** Every epoch is logged to MLflow.

In [ ]:
import mlflow
# SQLite backend (newer MLflow versions disable the old file-based ./mlruns store by default)
mlflow.set_tracking_uri(f"sqlite:///{OUT}/mlflow.db")
if mlflow.get_experiment_by_name("dermabench") is None:
    mlflow.create_experiment("dermabench", artifact_location="file://" + os.path.abspath(f"{OUT}/mlartifacts"))
mlflow.set_experiment("dermabench")

MEAN_T = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD_T = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class ArrDS(Dataset):
    def __init__(self, X, y, train=False):
        self.X, self.y, self.train = X, y, train
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(np.ascontiguousarray(self.X[i])).permute(2, 0, 1).float().div_(255)
        if self.train:   # dermoscopy has no canonical orientation -> flips / 90-degree rotations are label-preserving
            if torch.rand(1) < 0.5: x = x.flip(2)
            if torch.rand(1) < 0.5: x = x.flip(1)
            x = torch.rot90(x, int(torch.randint(0, 4, (1,))), (1, 2))
            x = (x * (0.9 + 0.2 * torch.rand(1))).clamp_(0, 1)
        return (x - MEAN_T) / STD_T, int(self.y[i])

def make_model(name, pretrained=True):
    import timm
    kw = dict(pretrained=pretrained and not SMOKE_TEST, num_classes=K)
    if name.startswith("vit"): kw["img_size"] = CFG["img_size"]
    return timm.create_model(name, **kw)

@torch.no_grad()
def predict_logits(model, X, bs=128):
    model.eval(); outs = []
    dl = DataLoader(ArrDS(X, np.zeros(len(X), dtype=int)), batch_size=bs, shuffle=False, num_workers=CFG["num_workers"])
    for xb, _ in dl:
        with torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=USE_AMP):
            out = model(xb.to(DEVICE, non_blocking=True))
        outs.append(out.float().cpu().numpy())
    return np.concatenate(outs)

def ckpt_path(name): return f"{OUT}/checkpoints/{name}.pt"

def train_model(name):
    hist_path = f"{OUT}/tables/train_history_{name}.json"
    if CFG["reuse_checkpoints"] and os.path.exists(ckpt_path(name)) and os.path.exists(hist_path):
        print(f"[{name}] checkpoint found, skipping training"); return json.load(open(hist_path))
    set_seed(CFG["seed"])
    model = make_model(name).to(DEVICE)
    counts = np.bincount(ytr, minlength=K); w = (len(ytr) / (K * counts)) ** 0.5; w = w / w.mean()
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    gen = torch.Generator(); gen.manual_seed(CFG["seed"])
    dl = DataLoader(ArrDS(Xtr, ytr, train=True), batch_size=CFG["batch_size"], shuffle=True, drop_last=True,
                    num_workers=CFG["num_workers"], pin_memory=(DEVICE == "cuda"), generator=gen)
    history, best = [], -1.0
    with mlflow.start_run(run_name=f"train-{name}"):
        mlflow.log_params({"model": name, "img_size": CFG["img_size"], "epochs": CFG["epochs"], "batch_size": CFG["batch_size"],
                           "lr": CFG["lr"], "weight_decay": CFG["weight_decay"], "seed": CFG["seed"], "loss": "weighted_CE_sqrt_inv_freq"})
        for ep in range(1, CFG["epochs"] + 1):
            model.train(); t0 = time.time(); tot = 0.0; n = 0
            for xb, yb in dl:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=USE_AMP):
                    loss = crit(model(xb), yb)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tot += loss.item() * len(yb); n += len(yb)
            sched.step()
            pv = softmax_np(predict_logits(model, Xva))
            va_auc, va_bal = METRIC_FNS["macro_auroc"](pv, yva), METRIC_FNS["bal_acc"](pv, yva)
            rec = dict(epoch=ep, train_loss=tot / n, val_macro_auroc=va_auc, val_bal_acc=va_bal, val_mel_sens=METRIC_FNS["mel_sens"](pv, yva), sec=time.time() - t0)
            history.append(rec); mlflow.log_metrics({k: v for k, v in rec.items() if k != "epoch"}, step=ep)
            if va_auc > best:
                best = va_auc; torch.save(model.state_dict(), ckpt_path(name)); rec["saved"] = True
            print(f"[{name}] ep {ep:02d}/{CFG['epochs']} loss {rec['train_loss']:.3f} | val AUROC {va_auc:.3f} bal-acc {va_bal:.3f} mel-sens {rec['val_mel_sens']:.3f} | {rec['sec']:.0f}s" + ("  ← best" if rec.get("saved") else ""))
        mlflow.log_metric("best_val_macro_auroc", best)
    json.dump(history, open(hist_path, "w"), indent=2)
    del model; torch.cuda.empty_cache()
    return history

HIST = {name: train_model(name) for name in CFG["models"]}
BEST_VAL = {n: max(h, key=lambda r: r["val_macro_auroc"])["val_macro_auroc"] for n, h in HIST.items()}
PRIMARY = max(BEST_VAL, key=BEST_VAL.get)   # chosen on VALIDATION performance only
print("\nBest val macro-AUROC:", {k: round(v, 4) for k, v in BEST_VAL.items()}, "→ primary model:", PRIMARY)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for n, h in HIST.items():
    ax[0].plot([r["epoch"] for r in h], [r["train_loss"] for r in h], label=n)
    ax[1].plot([r["epoch"] for r in h], [r["val_macro_auroc"] for r in h], label=n)
ax[0].set_title("Training loss"); ax[1].set_title("Validation macro-AUROC"); ax[0].legend(); ax[1].legend()
savefig("02_training_curves")

## 7. Test-set evaluation and calibration
Metrics come with **bootstrap 95% confidence intervals**; with only ~2,000 test images and a few dozen examples in the rarest classes, point estimates alone would overstate certainty. Temperature scaling is fit on the **validation** set only.

In [ ]:
def load_model(name):
    m = make_model(name, pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path(name), map_location=DEVICE)); return m.eval()

RES = {}
for name in CFG["models"]:
    m = load_model(name)
    lv, lt = predict_logits(m, Xva), predict_logits(m, Xte)
    T = fit_temperature(lv, yva)
    RES[name] = dict(T=T, probs_raw=softmax_np(lt), probs=softmax_np(lt / T))
    del m; torch.cuda.empty_cache()

SUMMARY, CALIB_ROWS = {}, []
for name, r in RES.items():
    SUMMARY[name] = summary_table(r["probs"], yte, CFG["n_boot"]).set_index("metric")
    SUMMARY[name].to_csv(f"{OUT}/tables/test_summary_{name}.csv")
    raw, ts = core_metrics(r["probs_raw"], yte), core_metrics(r["probs"], yte)
    CALIB_ROWS.append(dict(model=name, T=r["T"], ece_raw=raw["ece"], ece_ts=ts["ece"], nll_raw=raw["nll"], nll_ts=ts["nll"], brier_raw=raw["brier"], brier_ts=ts["brier"]))
CALIB = pd.DataFrame(CALIB_ROWS); CALIB.to_csv(f"{OUT}/tables/calibration_summary.csv", index=False)

def fmt(df, k): s = df.loc[k]; return f"{s.value:.3f} [{s.ci_low:.3f}-{s.ci_high:.3f}]"
comp = pd.DataFrame({n: {k: fmt(SUMMARY[n], k) for k in ["macro_auroc", "bal_acc", "acc", "mel_auroc", "mel_sens", "mel_spec", "ece"]} for n in SUMMARY}).T
comp.to_csv(f"{OUT}/tables/model_comparison.csv"); print("Test results (95% bootstrap CI), temperature-scaled probabilities:"); display(comp)
print("\nCalibration before/after temperature scaling:"); display(CALIB.round(4))

print(f"\nPrimary model (selected on validation): {PRIMARY}")
PC = per_class_table(RES[PRIMARY]["probs"], yte); PC.to_csv(f"{OUT}/tables/per_class_{PRIMARY}.csv", index=False); display(PC.round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
def reliability(ax, probs, y, title):
    conf, acc = probs.max(1), (probs.argmax(1) == y); edges = np.linspace(0, 1, CFG["ece_bins"] + 1); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mk = (conf > lo) & (conf <= hi)
        if mk.sum() > 0: xs.append(conf[mk].mean()); ys.append(acc[mk].mean())
    ax.plot([0, 1], [0, 1], "--", c="gray"); ax.plot(xs, ys, "o-"); ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_title(f"{title}\nECE={ece_score(probs, y):.3f}")
reliability(ax[0], RES[PRIMARY]["probs_raw"], yte, f"{PRIMARY}: raw")
reliability(ax[1], RES[PRIMARY]["probs"], yte, f"{PRIMARY}: temperature-scaled (T={RES[PRIMARY]['T']:.2f})")
ax[2].bar(PC.cls, PC.sensitivity); ax[2].set_title("Per-class sensitivity (recall)"); ax[2].set_ylim(0, 1)
for i, (s, n_) in enumerate(zip(PC.sensitivity, PC.n)): ax[2].text(i, s + 0.02, f"n={n_}", ha="center", fontsize=8)
savefig("03_calibration_and_per_class")

## 8. Proxy-subgroup analysis
Performance by image-quality band (thresholds from the **training** set). A subgroup gap that is large *and* statistically meaningful would be a finding to investigate; small subgroups are flagged and excluded from the gap criterion.

In [ ]:
def subgroup_table(probs, y, bands):
    pred = probs.argmax(1); overall = balanced_accuracy_score(y, pred); rows = []
    for feat in FEATURES:
        for band in ["low", "mid", "high"]:
            m = (bands[feat + "_band"] == band).values; n = int(m.sum())
            r = dict(feature=feat, band=band, n=n, mel_n=int((y[m] == MEL).sum()))
            if n > 0:
                r["acc"] = float((pred[m] == y[m]).mean()); r["bal_acc"] = float(balanced_accuracy_score(y[m], pred[m]))
                r["mel_sens"] = float(sens_spec_prec(pred[m], y[m], MEL)[0]) if r["mel_n"] >= CFG["min_class_n_subgroup"] else np.nan
                r["ece"] = ece_score(probs[m], y[m]); r["gap_vs_overall"] = overall - r["bal_acc"]
            r["used_for_criterion"] = n >= CFG["min_subgroup_n"]
            rows.append(r)
    return pd.DataFrame(rows), overall

SUBG, MAX_GAP = {}, {}
for name in CFG["models"]:
    SUBG[name], ov = subgroup_table(RES[name]["probs"], yte, BANDS_TE)
    MAX_GAP[name] = float(SUBG[name].query("used_for_criterion")["gap_vs_overall"].max())
    SUBG[name].to_csv(f"{OUT}/tables/subgroups_{name}.csv", index=False)
print("Max balanced-accuracy gap (overall − subgroup):", {k: round(v, 3) for k, v in MAX_GAP.items()})
display(SUBG[PRIMARY].round(3))

ov = balanced_accuracy_score(yte, RES[PRIMARY]["probs"].argmax(1))
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, feat in zip(axes, FEATURES):
    d = SUBG[PRIMARY][SUBG[PRIMARY].feature == feat]
    ax.bar(d.band, d.bal_acc); ax.axhline(ov, ls="--", c="red"); ax.set_title(feat)
    for i, (b, n_) in enumerate(zip(d.bal_acc, d.n)): ax.text(i, b + 0.01, f"n={n_}", ha="center", fontsize=8)
axes[0].set_ylabel("balanced accuracy"); fig.suptitle(f"{PRIMARY}: balanced accuracy by proxy subgroup (red = overall)")
savefig("04_proxy_subgroups")

## 9. Robustness to corruption
Five image corruptions at three severities, plus a **missing-colour-channel** test. The channel test is only a *crude analogue* of the "missing modality" problem. True missing-modality robustness needs multi-sequence data such as brain MRI (T1/T1c/T2/FLAIR), which is a separate study. We log it as exploratory and exclude it from the pass/fail criteria.

In [ ]:
def eval_perturbed(name, model, Xp, T):
    probs = softmax_np(predict_logits(model, Xp) / T)
    m = core_metrics(probs, yte); return {k: m[k] for k in ["bal_acc", "mel_sens", "macro_auroc", "ece"]}

rob_rows = []
for name in CFG["models"]:
    m = load_model(name); T = RES[name]["T"]; clean = core_metrics(RES[name]["probs"], yte)
    rob_rows.append(dict(model=name, kind="clean", level="clean", param="-", **{k: clean[k] for k in ["bal_acc", "mel_sens", "macro_auroc", "ece"]}))
    for kind, params in PERTURBATIONS.items():
        for lvl_name, p in zip(LEVEL_NAMES, params):
            rob_rows.append(dict(model=name, kind=kind, level=lvl_name, param=p, **eval_perturbed(name, m, perturb(Xte, kind, p), T)))
    for ch in CHANNELS:
        rob_rows.append(dict(model=name, kind="drop_channel", level=ch, param=ch, **eval_perturbed(name, m, perturb(Xte, "drop_channel", ch), T)))
    del m; torch.cuda.empty_cache(); print(f"[{name}] robustness done")
ROB = pd.DataFrame(rob_rows)
base = ROB[ROB.kind == "clean"].set_index("model")
ROB["mel_sens_drop"] = ROB.apply(lambda r: base.loc[r.model, "mel_sens"] - r.mel_sens, axis=1)
ROB["bal_acc_drop"] = ROB.apply(lambda r: base.loc[r.model, "bal_acc"] - r.bal_acc, axis=1)
ROB.to_csv(f"{OUT}/tables/robustness.csv", index=False)
display(ROB[ROB.model == PRIMARY].round(3))

MOD = ROB[(ROB.level == "moderate")]
MAX_MOD_DROP = MOD.groupby("model")["mel_sens_drop"].max().to_dict()
print("Largest melanoma-sensitivity drop at MODERATE severity:", {k: round(v, 3) for k, v in MAX_MOD_DROP.items()})

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
d = ROB[ROB.model == PRIMARY]; cl = d[d.kind == "clean"].iloc[0]
for kind in PERTURBATIONS:
    s = d[d.kind == kind]; xs = range(1, 4)
    ax[0].plot(xs, s.mel_sens.values, "o-", label=kind); ax[1].plot(xs, s.bal_acc.values, "o-", label=kind)
for a, k, t in [(ax[0], "mel_sens", "Melanoma sensitivity"), (ax[1], "bal_acc", "Balanced accuracy")]:
    a.axhline(cl[k], ls="--", c="k", label="clean"); a.set_xticks([1, 2, 3]); a.set_xticklabels(LEVEL_NAMES); a.set_title(t)
ax[0].legend(fontsize=7)
s = d[d.kind == "drop_channel"]; ax[2].bar(s.param, s.mel_sens, color=["tab:red", "tab:green", "tab:blue"]); ax[2].axhline(cl.mel_sens, ls="--", c="k")
ax[2].set_title("Melanoma sensitivity with one RGB channel removed")
fig.suptitle(f"{PRIMARY}: robustness"); savefig("05_robustness")

## 10. Failure-mode analysis (primary model)
We look at *where and why* the model fails rather than only how often: confusion structure, error rates by proxy subgroup, the most confident mistakes, and **EigenCAM / Grad-CAM** overlays for those mistakes. Saliency maps are hypothesis-generating tools, not proof of what the model "uses", so we treat them accordingly.

In [ ]:
probs = RES[PRIMARY]["probs"]; pred = probs.argmax(1); conf = probs.max(1)
FAIL = pd.DataFrame(dict(idx=np.arange(len(yte)), true=yte, pred=pred, conf=conf, correct=pred == yte)).join(BANDS_TE)
def etype(r):
    if r.correct: return "correct"
    if r.true == MEL: return "missed melanoma (FN)"
    if r.pred == MEL: return "false melanoma alarm (FP)"
    return "other confusion"
FAIL["error_type"] = FAIL.apply(etype, axis=1)
FAIL.to_csv(f"{OUT}/tables/failure_table_{PRIMARY}.csv", index=False)
print(FAIL.error_type.value_counts().to_string())

cm = confusion_matrix(yte, pred, labels=range(K)); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax[0].set_xticks(range(K)); ax[0].set_yticks(range(K))
ax[0].set_xticklabels(CLASS_NAMES, rotation=45); ax[0].set_yticklabels(CLASS_NAMES); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
for i in range(K):
    for j in range(K): ax[0].text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
ax[0].set_title("Row-normalised confusion matrix")
er = [(f"{f}:{b}", 1 - FAIL[FAIL[f + "_band"] == b].correct.mean()) for f in FEATURES for b in ["low", "mid", "high"]]
ax[1].barh([e[0] for e in er], [e[1] for e in er]); ax[1].axvline(1 - FAIL.correct.mean(), ls="--", c="red"); ax[1].set_title("Error rate by proxy subgroup (red = overall)")
savefig("06_confusion_and_error_rates")

mel_fn = FAIL[FAIL.error_type == "missed melanoma (FN)"]
print(f"\nMissed melanomas: {len(mel_fn)}. Where did they go?"); print(pd.Series(CLASS_NAMES)[mel_fn.pred.values].value_counts().to_string())

In [ ]:
from pytorch_grad_cam import EigenCAM, GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def cam_layers(name, model):
    if name.startswith("vit"):
        ps = model.patch_embed.patch_size; ps = ps[0] if isinstance(ps, (tuple, list)) else ps
        grid = CFG["img_size"] // ps
        def rt(t):
            t = t[:, -grid * grid:, :]                                   # drop cls/prefix tokens
            return t.reshape(t.size(0), grid, grid, t.size(2)).permute(0, 3, 1, 2)
        return [model.blocks[-1].norm1], rt
    return [model.layer4[-1]], None

model = load_model(PRIMARY); layers, rt = cam_layers(PRIMARY, model)
top = FAIL[~FAIL.correct].sort_values("conf", ascending=False).head(6)
imgs = Xte[top.idx.values]
xs = torch.stack([ArrDS(imgs, np.zeros(len(imgs), dtype=int))[i][0] for i in range(len(imgs))]).to(DEVICE)
with EigenCAM(model=model, target_layers=layers, reshape_transform=rt) as cam: eig = cam(input_tensor=xs, targets=None)
with GradCAM(model=model, target_layers=layers, reshape_transform=rt) as cam: gc = cam(input_tensor=xs, targets=[ClassifierOutputTarget(int(p)) for p in top.pred.values])

fig, axes = plt.subplots(3, len(top), figsize=(2.6 * len(top), 8))
for j, (_, r) in enumerate(top.iterrows()):
    rgb = imgs[j].astype(np.float32) / 255
    axes[0, j].imshow(rgb); axes[0, j].set_title(f"true {CLASS_NAMES[r.true]} → pred {CLASS_NAMES[r.pred]}\nconf {r.conf:.2f}", fontsize=8)
    axes[1, j].imshow(show_cam_on_image(rgb, eig[j], use_rgb=True)); axes[2, j].imshow(show_cam_on_image(rgb, gc[j], use_rgb=True))
for a in axes.ravel(): a.axis("off")
axes[1, 0].set_title("EigenCAM", fontsize=9, loc="left"); axes[2, 0].set_title("Grad-CAM (predicted class)", fontsize=9, loc="left")
fig.suptitle(f"{PRIMARY}: most confident errors", y=0.995); savefig("07_cam_confident_errors")
del model; torch.cuda.empty_cache()

# --- manual review sheet: YOU annotate the top-50 confident errors, then summarise the categories in the report
review = FAIL[~FAIL.correct].sort_values("conf", ascending=False).head(50)[["idx", "true", "pred", "conf", "error_type"]].copy()
review["true"] = review.true.map(lambda i: CLASS_NAMES[i]); review["pred"] = review.pred.map(lambda i: CLASS_NAMES[i])
review["reviewer_category"] = ""; review["notes"] = ""
review.to_csv(f"{OUT}/tables/failure_review_sheet.csv", index=False)
print("Saved failure_review_sheet.csv: fill in 'reviewer_category' (e.g. hair/occlusion, ruler or ink marks, low contrast, atypical morphology, ambiguous label) after eyeballing the images.")

### Your manual failure-mode review (fill this in after looking at the figure and the review sheet)
Write only what you actually observe. Suggested structure:
- **Pattern 1:** …(how many of the top-50 errors, which classes, what the CAMs highlight)
- **Pattern 2:** …
- **What I could *not* determine:** …
- **Proposed mitigation (and how I'd test it):** …

## 11. Acceptance-criteria verdict
Compare the **pre-declared** criteria with the measured results for the primary model. A fail is a valid outcome: the point of the exercise is an honest, traceable verdict.

In [ ]:
S = SUMMARY[PRIMARY]; crit_rows = []
def add(name, value, thr, op, ci=None):
    ok = value >= thr if op == ">=" else value <= thr
    note = ""
    if ci is not None and not np.isnan(ci[0]):
        crosses = (ci[0] < thr <= ci[1]) if op == ">=" else (ci[0] <= thr < ci[1])
        note = "CI straddles threshold: inconclusive" if crosses else ""
    crit_rows.append(dict(criterion=name, threshold=f"{op} {thr}", measured=round(float(value), 4), verdict="PASS" if ok else "FAIL", note=note))
add("macro AUROC", S.loc["macro_auroc", "value"], CRITERIA["macro_auroc_min"], ">=", (S.loc["macro_auroc", "ci_low"], S.loc["macro_auroc", "ci_high"]))
add("melanoma sensitivity", S.loc["mel_sens", "value"], CRITERIA["mel_sensitivity_min"], ">=", (S.loc["mel_sens", "ci_low"], S.loc["mel_sens", "ci_high"]))
add("ECE after temperature scaling", S.loc["ece", "value"], CRITERIA["ece_after_ts_max"], "<=")
add("max proxy-subgroup balanced-acc gap", MAX_GAP[PRIMARY], CRITERIA["max_subgroup_balacc_gap"], "<=")
add("max melanoma-sens drop, moderate corruptions", MAX_MOD_DROP[PRIMARY], CRITERIA["max_moderate_mel_sens_drop"], "<=")
ACCEPT = pd.DataFrame(crit_rows); ACCEPT.to_csv(f"{OUT}/tables/acceptance_verdict.csv", index=False)
display(ACCEPT)
print("OVERALL:", "ALL CRITERIA MET" if (ACCEPT.verdict == "PASS").all() else "NOT ALL CRITERIA MET (see FAIL rows)")

## 12. Log to MLflow, generate the model card, bundle outputs

In [ ]:
with mlflow.start_run(run_name="evaluation"):
    mlflow.log_params({"primary_model": PRIMARY, "img_size": CFG["img_size"], "n_boot": CFG["n_boot"], **{f"crit_{k}": v for k, v in CRITERIA.items()}})
    for k in ["macro_auroc", "bal_acc", "mel_sens", "mel_auroc", "ece"]:
        mlflow.log_metric(f"test_{k}", float(SUMMARY[PRIMARY].loc[k, "value"]))
    mlflow.log_metric("max_subgroup_gap", MAX_GAP[PRIMARY]); mlflow.log_metric("max_moderate_mel_sens_drop", MAX_MOD_DROP[PRIMARY])
    mlflow.log_artifacts(f"{OUT}/figures", "figures"); mlflow.log_artifacts(f"{OUT}/tables", "tables")

def mdt(df): return df.to_markdown(index=False)
card = f"""# Model Card: DermaBench ({PRIMARY})

## Intended use
Research / educational benchmark for a skin-lesion **triage decision-support** component. **Not a medical device. Not validated for clinical use.** Not for diagnosis.

## Data
DermaMNIST (MedMNIST+, derived from HAM10000), {CFG['img_size']}×{CFG['img_size']} px, 7 classes, fixed splits
(train {len(ytr)} / val {len(yva)} / test {len(yte)}). Data fingerprints are in `tables/data_fingerprint.json`.

## Model & training
`{PRIMARY}` (timm, ImageNet-pretrained, fine-tuned). Compared against: {", ".join(CFG["models"])}. AdamW, cosine schedule, {CFG['epochs']} epochs,
sqrt-inverse-frequency class weights, flip/rotation/brightness augmentation, seed {CFG['seed']}. Checkpoint chosen on validation macro-AUROC.
Temperature scaling (T = {RES[PRIMARY]['T']:.3f}) fitted on the validation set. Environment: `tables/run_config_environment.json`.

## Acceptance criteria (declared before test evaluation; `tables/criteria_frozen.json`)
{mdt(ACCEPT)}

## Test performance (95% bootstrap CI, {CFG['n_boot']} resamples)
{mdt(SUMMARY[PRIMARY].round(4).reset_index())}

### Per-class
{mdt(PC.round(3))}

### Model comparison
{mdt(comp.reset_index().rename(columns={'index': 'model'}))}

## Calibration
{mdt(CALIB.round(4))}

## Proxy-subgroup analysis (image-quality bands from training-set tertiles; NOT demographics)
{mdt(SUBG[PRIMARY].round(3))}

## Robustness (primary model)
{mdt(ROB[ROB.model == PRIMARY].round(3))}

## Failure modes
See `figures/06_confusion_and_error_rates.png`, `figures/07_cam_confident_errors.png`, `tables/failure_review_sheet.csv`.
*(Add your manually reviewed failure-mode summary here.)*

## Limitations
- **No demographics / patient identifiers** in MedMNIST: fairness across age, sex, skin tone or site could not be assessed. Proxy subgroups are image-quality bands only.
- **Lesion/patient-level split integrity cannot be verified**; HAM10000 contains multiple images of some lesions, so test performance may be optimistic.
- Images are **downsampled** from the originals; results may differ at native resolution.
- **Single-source dataset** (dermoscopy only); no external validation, no prospective or reader study.
- Corruption tests are synthetic; the missing-channel test is only a crude analogue of missing modalities.
- Saliency maps (EigenCAM/Grad-CAM) are qualitative and do not establish causal feature use.
- Results are from a single training seed; seed variance was not quantified.
"""
open(f"{OUT}/MODEL_CARD.md", "w").write(card)
print(card[:1500], "\n... (full card saved to MODEL_CARD.md)")

In [ ]:
import zipfile
bundle = f"{OUT}_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        if "checkpoints" in root or "mlartifacts" in root: continue   # weights excluded (large); mlartifacts duplicates figures/tables
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print("Bundle:", bundle, f"({os.path.getsize(bundle) / 1e6:.1f} MB)")
# On Kaggle the bundle is already in /kaggle/working, so download it from the notebook's Output tab after the run finishes
# (use "Save Version → Save & Run All (Commit)" so the outputs are kept).

## What to do next
1. Download the bundle from the Output tab. Put `MODEL_CARD.md`, `figures/` and `tables/` into your GitHub repo (`results/`).
2. Fill in the **failure-mode review** section of the model card with what you actually observed.
3. In the README, report the results **as measured**, including any failed acceptance criteria and the limitations.
4. Next step: refactor the helper library into `src/dermabench/`, add `pytest` tests, DVC for data/checkpoints, and a GitHub Actions workflow.